## Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import time
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack
from sklearn.metrics import root_mean_squared_error

from catboost import CatBoostRegressor

RANDOM_STATE = 42

/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Повторяю всё те же действия, чтобы проводить подбор на маленьком стратифицированном датасете


In [2]:
df_train = pd.read_csv("../train.csv")

Тк стратифицированная подвыборка работает только для классификации, поделю целевой признак на 100 классов

In [3]:
_y = (df_train['deal_probability'].round(2) * 100).astype(int)

In [4]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=300000, random_state=42)

In [5]:
rest_idx, small_idx = next(splitter.split(df_train, _y))
df_small = df_train.iloc[small_idx].reset_index(drop=True)

Отделяю по 10% валидационную и тестовую выборки, отложенные по времени 

In [6]:
df_small['activation_date'] = pd.to_datetime(df_small['activation_date'])
df_small = df_small.sort_values(by='activation_date').reset_index(drop=True)

length = len(df_small.index)
train_end_id = int(0.8 * length)
valid_end_id = int(0.9 * length)

df_small_train = df_small.iloc[:train_end_id, :]
df_small_valid = df_small.iloc[train_end_id:valid_end_id, :]
df_small_test = df_small.iloc[valid_end_id:, :]

## Подбор ГП с optuna

In [36]:
def gradient_text_preprocessing_fit_transform(df_train):
    df = df_train.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)
    df['title'] = df['title'].astype(str)
    df['description'] = df['description'].astype(str)

    cat_features = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_features:
        df[col] = df[col].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'item_seq_number', 'title', 'description']
    
    X_train = df[feature_names]
    y_train = df['deal_probability']

    return X_train, y_train, feature_names


def gradient_text_preprocessing_transform(df_valid, feature_names):
    df = df_valid.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)
    df['title'] = df['title'].astype(str)
    df['description'] = df['description'].astype(str)
    
    cat_features = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_features:
        df[col] = df[col].astype(str)
    
    X_valid = df[feature_names]
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [37]:
static_params = {
    'iterations': 1500,
    'cat_features': ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'],
    'text_features': ['title', 'description'],
    'early_stopping_rounds': 20,
    'verbose': False,
    'random_seed': 42,
    'thread_count': -1
}
tscv = TimeSeriesSplit(n_splits=5)
def objective(trial):

    dynamic_params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 2, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    model_params = {**static_params, **dynamic_params}
    
    scores = []

    for train_idx, val_idx in tscv.split(df_small_train):
        df_tr, df_val = df_small_train.iloc[train_idx], df_small_train.iloc[val_idx]

        n = len(df_tr)
        cut = int(n * 0.9)
        df_tr_fit, df_tr_es = df_tr.iloc[:cut], df_tr.iloc[cut:]
        
        X_tr, y_tr, feature_names = gradient_text_preprocessing_fit_transform(df_tr_fit)
        X_vl, y_vl = gradient_text_preprocessing_transform(df_val, feature_names)
        X_val_eval, y_val_eval = gradient_text_preprocessing_transform(df_tr_es, feature_names)
        
        model = CatBoostRegressor(**model_params)
        model.fit(X_tr, y_tr, eval_set=(X_val_eval, y_val_eval))
        
        preds = model.predict(X_vl)
        fold_rmse = root_mean_squared_error(y_vl, preds)
        scores.append(fold_rmse)
        print(f"----Получили RMSE на фолде: {fold_rmse}----")
        
    return np.mean(scores)




In [19]:
%%time

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print("Лучшие параметры:", study.best_params)
print("Лучшее RMSE:", study.best_value)

[I 2026-10-14 21:15:23,139] A new study created in memory with name: no-name-20d62eab-8a48-41cb-80f9-0d169b303668


----Получили RMSE на фолде: 0.23409226080726145----
----Получили RMSE на фолде: 0.23453550608321014----
----Получили RMSE на фолде: 0.22955336563909212----
----Получили RMSE на фолде: 0.2310051324002025----


[I 2026-10-14 21:23:55,472] Trial 0 finished with value: 0.23249672319538028 and parameters: {'learning_rate': 0.014925364078554989, 'depth': 4, 'l2_leaf_reg': 8.025440818764395, 'random_strength': 1.707325726062627, 'bagging_temperature': 0.25220235444687955}. Best is trial 0 with value: 0.23249672319538028.


----Получили RMSE на фолде: 0.23329735104713512----
----Получили RMSE на фолде: 0.23368779687936933----
----Получили RMSE на фолде: 0.23365429590354425----
----Получили RMSE на фолде: 0.2291474270755458----
----Получили RMSE на фолде: 0.2293955908253091----


[I 2026-10-14 21:28:27,411] Trial 1 finished with value: 0.23143867873945206 and parameters: {'learning_rate': 0.0867473573521225, 'depth': 3, 'l2_leaf_reg': 1.350029619859539, 'random_strength': 0.7556999997636344, 'bagging_temperature': 0.2612788049630257}. Best is trial 1 with value: 0.23143867873945206.


----Получили RMSE на фолде: 0.23130828301349168----
----Получили RMSE на фолде: 0.23359027453933504----
----Получили RMSE на фолде: 0.23345496907175903----
----Получили RMSE на фолде: 0.22829657792565994----
----Получили RMSE на фолде: 0.22969728406093495----


... (32 outputs left out)

[I 2026-10-15 14:27:08,910] Trial 18 finished with value: 0.229432508677736 and parameters: {'learning_rate': 0.09534892976599847, 'depth': 10, 'l2_leaf_reg': 8.855857568073043, 'random_strength': 1.329983305425388, 'bagging_temperature': 0.1564699846601455}. Best is trial 17 with value: 0.22917339990244492.


----Получили RMSE на фолде: 0.22929229702088375----
----Получили RMSE на фолде: 0.2320671056086807----
----Получили RMSE на фолде: 0.2319768076436863----
----Получили RMSE на фолде: 0.22663662344763122----
----Получили RMSE на фолде: 0.2271379003085423----


[I 2026-10-15 14:50:45,142] Trial 19 finished with value: 0.22949306917689077 and parameters: {'learning_rate': 0.09229407145766798, 'depth': 9, 'l2_leaf_reg': 5.22066877261137, 'random_strength': 1.1766043579753753, 'bagging_temperature': 0.41152364942733144}. Best is trial 17 with value: 0.22917339990244492.


----Получили RMSE на фолде: 0.22964690887591324----
Лучшие параметры: {'learning_rate': 0.08047291588307089, 'depth': 10, 'l2_leaf_reg': 5.851792917605791, 'random_strength': 1.5837729868972885, 'bagging_temperature': 0.23593153067360195}
Лучшее RMSE: 0.22917339990244492
CPU times: user 6d 17h 8min 49s, sys: 2h 29min 16s, total: 6d 19h 38min 5s
Wall time: 17h 35min 22s


Это было мягко говоря долго, потому что я забыла включить распараллелить проццесс ('thread_count': -1)

Итого лучшие параметры: {'learning_rate': 0.08047291588307089, 'depth': 10, 'l2_leaf_reg': 5.851792917605791, 'random_strength': 1.5837729868972885, 'bagging_temperature': 0.23593153067360195}
Лучшее RMSE: 0.22917339990244492

In [38]:
best_params = {**static_params, **study.best_params}

In [39]:
X_tr1, y_tr1, feature_names = gradient_text_preprocessing_fit_transform(df_small_train)
X_vl1, y_vl1 = gradient_text_preprocessing_transform(df_small_test, feature_names)


In [40]:
catbost = CatBoostRegressor(**best_params)
catbost.fit(X_tr1, y_tr1)

CatBoostRegressor(bagging_temperature=0.23593153067360195, cat_features=['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], depth=10, early_stopping_rounds=20, iterations=1500, l2_leaf_reg=5.851792917605791, learning_rate=0.08047291588307089, loss_function='RMSE', random_seed=42, random_strength=1.5837729868972885, text_features=['title', 'description'], verbose=False)

In [41]:
preds1 = catbost.predict(X_vl1)
print(root_mean_squared_error(y_vl1, preds1))

0.22260539825133252


Засчет подбора ГП смогли улучить RMSE c 0.227 до 0.226!!!

## Обучим финальную модель на полном датасете:

In [42]:
df_test = pd.read_csv("../test.csv")

In [47]:
X_train, y_train, feature_names = gradient_text_preprocessing_fit_transform(df_train)
X_test, y_test = gradient_text_preprocessing_transform(df_test, feature_names)

In [44]:
catbost.fit(X_train, y_train)

CatBoostRegressor(bagging_temperature=0.23593153067360195, cat_features=['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], depth=10, early_stopping_rounds=20, iterations=1500, l2_leaf_reg=5.851792917605791, learning_rate=0.08047291588307089, loss_function='RMSE', random_seed=42, random_strength=1.5837729868972885, text_features=['title', 'description'], verbose=False)

In [48]:
X_test

,region,city,parent_category_name,category_name,param_1,activation_day_of_week,user_type,image_top_1,price,item_seq_number,title,description
0,Волгоградская область,Волгоград,Личные вещи,Детская одежда и обувь,Для мальчиков,1,Private,2020.0,NaN,66,Отдам бесплатно,На ангарском
1,Свердловская область,Нижняя Тура,Хобби и отдых,Велосипеды,Дорожные,6,Private,nan,3000.0,4,Продам велосипед,"Продам велосипед KAMA F200,в нормальном состо..."
2,Новосибирская область,Бердск,Бытовая электроника,Аудио и видео,Телевизоры и проекторы,0,Private,2960.0,15000.0,15,BBK,Продам новый телевизор BBK 32 диагональ смарт...
3,Саратовская область,Саратов,Для дома и дачи,Бытовая техника,Для кухни,0,Private,nan,4500.0,70,Вытяжка Jetair 60,"Продам новую вытяжку в упаковке,с документами...."
4,Оренбургская область,Бузулук,Личные вещи,Товары для детей и игрушки,Детские коляски,5,Private,1002.0,4900.0,15,Коляска зима-лето,Продам отличную коляску. б/у 1 год. все вопрос...
...,...,...,...,...,...,...,...,...,...,...,...,...
508433,Кемеровская область,Кемерово,Бытовая электроника,Телефоны,Nokia,1,Private,2939.0,1200.0,39,Nokia Asha,"Продам телефон Nokia Asha, черный, две симки, ..."
508434,Тульская область,Тула,Бытовая электроника,Ноутбуки,nan,2,Company,2951.0,28000.0,168,Игровой ноутбук Asus G56JK-CN128H,Состояние хорошее. Жесткий диск на 1 тр.
508435,Алтайский край,Барнаул,Личные вещи,"Одежда, обувь, аксессуары",Женская одежда,4,Private,572.0,900.0,112,Новое платье Bizzarro,"платье Bizzarro новое c этикеткой, куплено по..."
508436,Ставропольский край,Невинномысск,Услуги,Предложение услуг,"Оборудование, производство",3,Private,nan,NaN,1,Профтлактика и ремонт сплит систем,Выполню ремонт и обслуживание сплит систем нед...


## Финальное предсказание:

In [49]:
final_preds = catbost.predict(X_test)
final_preds = np.clip(final_preds, 0, 1)

In [50]:
submission = pd.DataFrame({ 
    'item_id': df_test['item_id'],
    'deal_probability': final_preds
})
submission.to_csv('avito_first_submission.csv', index=False)

После сабмита на Kaggle:
- Public Score RMSE=0.22659
- Private Score RMSE=0.22994